In [1]:
# skip this cell if you already downloaded the data
!git clone https://github.com/Belaleatsbanana/dl.git
%cd dl
!python get_data.py

Cloning into 'dl'...
remote: Enumerating objects: 81, done.
remote: Counting objects: 100% (81/81), done.
remote: Compressing objects: 100% (50/50), done.
remote: Total 81 (delta 39), reused 73 (delta 31), pack-reused 0 (from 0)
Receiving objects: 100% (81/81), 40.93 KiB | 20.47 MiB/s, done.
Resolving deltas: 100% (39/39), done.
/content/dl
Using Colab cache for faster access to the 'food-101' dataset.
Copied: train/beef_tartare
Copied: train/chicken_quesadilla
Copied: train/risotto
Copied: train/spaghetti_carbonara
Copied: train/pancakes
Copied: validation/beef_tartare
Copied: validation/chicken_quesadilla
Copied: validation/risotto
Copied: validation/spaghetti_carbonara
Copied: validation/pancakes


In [3]:
!python main.py --model_to_run resnext50_local

/usr/local/lib/python3.12/dist-packages/timm/models/registry.py:4: FutureWarning: Importing from timm.models.registry is deprecated, please import via timm.models
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.models", FutureWarning)
/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
/content/dl/DeiT/DeiT.py:68: UserWarning: Overwriting deit_tiny_distilled_patch16_224 in registry with DeiT.DeiT.deit_tiny_distilled_patch16_224. This is because the name being registered conflicts with an existing name. Please check if this is not expected.
  @register_model
[2026-03-31 18:15:47,128 resnext50_local] (main.py 21): INFO Dataset loaded: 4750 train images, 250 validation images
[2026-03-31 18:15:47,128 resnext50_local] (main.py 23): INFO 

In [ ]:
!python main.py --model_to_run deit_tiny

In [ ]:
!python main.py --model_to_run as_mlp_tiny

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from models import get_model_config

def load_results(output_dir='outputs'):
    metrics_dfs = []
    cms = {}
    best_metrics = []
    
    if not os.path.exists(output_dir):
        return None, None, None
        
    files = os.listdir(output_dir)
    model_names = set([f.replace('_metrics.csv', '') for f in files if f.endswith('_metrics.csv')])
    
    for name in model_names:
        # Load metrics
        metrics_path = os.path.join(output_dir, f'{name}_metrics.csv')
        df = pd.read_csv(metrics_path)
        
        # Get TAG
        config = get_model_config(name)
        tag = config['MODEL']['TAG']
        df['model_name'] = name
        df['tag'] = tag
        metrics_dfs.append(df)
        
        # Get best metrics
        best_epoch_idx = df['val_acc'].idxmax()
        best_row = df.loc[best_epoch_idx].copy()
        best_metrics.append(best_row)
        
        # Load confusion matrix
        cm_path = os.path.join(output_dir, f'{name}_confusion_matrix.npy')
        if os.path.exists(cm_path):
            cms[name] = np.load(cm_path)
            
    all_metrics = pd.concat(metrics_dfs, ignore_index=True)
    best_metrics_df = pd.DataFrame(best_metrics)
    
    return all_metrics, best_metrics_df, cms

all_metrics, best_metrics_df, cms = load_results()
if best_metrics_df is not None:
    display(best_metrics_df.sort_values(by='val_acc', ascending=False))

## Learning Curves

In [ ]:
def plot_learning_curves(df, tag):
    tag_df = df[df['tag'] == tag]
    if tag_df.empty:
        print(f"No results for {tag}")
        return
    
    plt.figure(figsize=(15, 5))
    
    # Accuracy plot
    plt.subplot(1, 2, 1)
    for name in tag_df['model_name'].unique():
        model_df = tag_df[tag_df['model_name'] == name]
        plt.plot(model_df['epoch'], model_df['val_acc'], label=f'{name} (val)')
        plt.plot(model_df['epoch'], model_df['train_acc'], '--', alpha=0.5, label=f'{name} (train)')
    plt.title(f'{tag} Accuracy Curves')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy (%)')
    plt.legend()
    plt.grid(True)
    
    # Loss plot
    plt.subplot(1, 2, 2)
    for name in tag_df['model_name'].unique():
        model_df = tag_df[tag_df['model_name'] == name]
        plt.plot(model_df['epoch'], model_df['val_loss'], label=f'{name} (val)')
        plt.plot(model_df['epoch'], model_df['train_loss'], '--', alpha=0.5, label=f'{name} (train)')
    plt.title(f'{tag} Loss Curves')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)
    
    plt.tight_layout()
    plt.show()

plot_learning_curves(all_metrics, 'CNN')

In [ ]:
plot_learning_curves(all_metrics, 'MLP')

In [ ]:
plot_learning_curves(all_metrics, 'Transformer')

## Confusion Matrices Comparison

In [ ]:
def plot_cms(best_df, cms_dict, tag):
    tag_models = best_df[best_df['tag'] == tag]['model_name'].tolist()
    if not tag_models:
        print(f"No confusion matrices for {tag}")
        return
        
    n_models = len(tag_models)
    plt.figure(figsize=(6*n_models, 5))
    
    for i, name in enumerate(tag_models):
        if name in cms_dict:
            plt.subplot(1, n_models, i+1)
            sns.heatmap(cms_dict[name], annot=True, fmt='d', cmap='Blues')
            plt.title(f'CM: {name}')
            plt.xlabel('Predicted')
            plt.ylabel('True')
            
    plt.tight_layout()
    plt.show()

plot_cms(best_metrics_df, cms, 'CNN')

In [ ]:
plot_cms(best_metrics_df, cms, 'MLP')

In [ ]:
plot_cms(best_metrics_df, cms, 'Transformer')

## Final Comparison: Best CNN vs Best MLP vs Best Transformer

In [ ]:
def final_comparison(best_df, cms_dict):
    # Get best model for each tag
    final_best = best_df.sort_values('val_acc', ascending=False).groupby('tag').head(1)
    display(final_best[['tag', 'model_name', 'val_acc', 'val_f1', 'val_precision', 'val_recall']])
    
    # Plot their CMs
    best_models = final_best['model_name'].tolist()
    plt.figure(figsize=(6*len(best_models), 5))
    for i, name in enumerate(best_models):
        if name in cms_dict:
            plt.subplot(1, len(best_models), i+1)
            sns.heatmap(cms_dict[name], annot=True, fmt='d', cmap='Greens')
            tag = final_best[final_best['model_name'] == name]['tag'].values[0]
            plt.title(f'Best {tag}: {name}')
            plt.xlabel('Predicted')
            plt.ylabel('True')
    plt.tight_layout()
    plt.show()

if best_metrics_df is not None:
    final_comparison(best_metrics_df, cms)

In [ ]:
import shutil
shutil.make_archive('outputs', 'zip', 'outputs')
print("Outputs zipped to outputs.zip")